In [9]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.preprocessing import OrdinalEncoder
from scipy.stats import randint, uniform

In [14]:
train_df = pd.read_csv("join_train.csv")
test_df = pd.read_csv("join_test.csv")
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

In [15]:
categorical_cols = X.select_dtypes(include=['object']).columns
encoder_dict = {}
for col in categorical_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[col] = encoder.fit_transform(X[[col]])
    test_df[col] = encoder.transform(test_df[[col]])
    encoder_dict[col] = encoder

In [16]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [17]:
xgb_model = XGBClassifier(
    use_label_encoder=False,
    njob=-1,
    eval_metric='auc',
    random_state=42
)
xgb_model.fit(X_train, y_train)

y_train_proba = xgb_model.predict_proba(X_train)[:, 1]
y_train_pred = (y_train_proba > 0.5).astype(int)
print("=== Train ===")
print(classification_report(y_train, y_train_pred))
print("AUC PR:", average_precision_score(y_train, y_train_proba))
print("ROC AUC:", roc_auc_score(y_train, y_train_proba))

y_val_proba = xgb_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)
print("=== validation ===")
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [13:25:59] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "njob", "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


=== Train ===
              precision    recall  f1-score   support

           0       0.77      0.97      0.86    152025
           1       0.63      0.17      0.27     52973

    accuracy                           0.76    204998
   macro avg       0.70      0.57      0.56    204998
weighted avg       0.73      0.76      0.70    204998

AUC PR: 0.5158344073323591
ROC AUC: 0.7673421503589621
=== validation ===
              precision    recall  f1-score   support

           0       0.76      0.96      0.85     38006
           1       0.52      0.14      0.22     13244

    accuracy                           0.74     51250
   macro avg       0.64      0.55      0.53     51250
weighted avg       0.70      0.74      0.68     51250

AUC PR: 0.4370568705100318
ROC AUC: 0.7331279392086958


## 랜덤 서치

In [112]:
train_df = pd.read_csv("train_e.csv")
test_df = pd.read_csv("test_e.csv")
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
xgb_model = XGBClassifier(
    use_label_encoder=False, n_jobs=-1,
    eval_metric='auc', random_state=42)

param_list = { 
    'n_estimators': randint(50, 1000),
    'learning_rate': uniform(0.001, 0.299),
    'max_depth': randint(2, 11),
    'subsample': uniform(0.5, 0.5),        # 0.5 ~ 1.0
    'colsample_bytree': uniform(0.5, 0.5), # 0.5 ~ 1.0
    'gamma': uniform(0, 0.5),             # 0.0 ~ 0.5
    'min_child_weight': randint(1, 10),
    'reg_alpha': uniform(0, 1.0),         # 0.0 ~ 1.0
    'reg_lambda': uniform(0, 2.0),        # 0.0 ~ 2.0
    'scale_pos_weight': uniform(0.5, 4.5)
}
scv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_list,
    n_iter=50, 
    scoring='roc_auc', 
    cv=scv,  
    verbose=2,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)
print("Best Parameters:", random_search.best_params_)

best_xgb = random_search.best_estimator_
y_train_proba = best_xgb.predict_proba(X_train)[:, 1]
y_train_pred = (y_train_proba > 0.5).astype(int)
print("=== Train ===")
print(classification_report(y_train, y_train_pred))
print("AUC PR:", average_precision_score(y_train, y_train_proba))
print("ROC AUC:", roc_auc_score(y_train, y_train_proba))

y_val_proba = best_xgb.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)
print("=== validation ===")
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:08:58] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Parameters: {'colsample_bytree': 0.8288064461501716, 'gamma': 0.2841543016677358, 'learning_rate': 0.02900875558059965, 'max_depth': 4, 'min_child_weight': 6, 'n_estimators': 424, 'reg_alpha': 0.25046181860558414, 'reg_lambda': 1.1797416951210877, 'scale_pos_weight': 4.905017862237541, 'subsample': 0.7433710764797276}
=== Train ===
              precision    recall  f1-score   support

           0       0.95      0.37      0.53    150921
           1       0.34      0.94      0.50     52841

    accuracy                           0.52    203762
   macro avg       0.65      0.65      0.52    203762
weighted avg       0.79      0.52      0.52    203762

AUC PR: 0.4587346700838873
ROC AUC: 0.7425310649098452
=== validation ===
              precision    recall  f1-score   support

           0       0.95      0.37      0.53     37730
           1       0.34      0.94      0.50     13211

    accuracy                           0.52     50941
   macro avg       0.64      0.65      0.5

In [116]:
xgb_model = XGBClassifier(
    use_label_encoder=False, n_jobs=-1,
    eval_metric='auc', random_state=42)

param_grid = {
    'n_estimators':       [400, 424, 450],   # 최적(424) 주변
    'learning_rate':      [0.02, 0.03],  # 최적(0.029) 주변
    'max_depth':          [4],         # 최적(4) 주변
    'min_child_weight':   [6],         # 최적(6) 주변
    'gamma':              [0.28, 0.3], # 최적(0.284) 주변
    'subsample':          [0.74, 0.78], # 최적(0.743) 주변
    'colsample_bytree':   [0.8, 0.83], # 최적(0.829) 주변
    'reg_alpha':          [0.25],  # 최적(0.25) 주변
    'reg_lambda':         [1.2],   # 최적(1.18) 주변
    'scale_pos_weight':   [4, 5] 
}
scv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=xgb_model, 
                           param_grid=param_grid,
                           scoring='roc_auc',
                           cv=scv,
                           verbose=1,
                           n_jobs=-1)

grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
best_xgb = grid_search.best_estimator_
y_train_proba = best_xgb.predict_proba(X_train)[:, 1]
y_train_pred = (y_train_proba > 0.5).astype(int)
print("=== Train ===")
print(classification_report(y_train, y_train_pred))
print("AUC PR:", average_precision_score(y_train, y_train_proba))
print("ROC AUC:", roc_auc_score(y_train, y_train_proba))

y_val_proba = grid_search.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)
print("=== validation ===")
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

Fitting 5 folds for each of 96 candidates, totalling 480 fits


d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:42:53] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Parameters: {'colsample_bytree': 0.8, 'gamma': 0.28, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 6, 'n_estimators': 450, 'reg_alpha': 0.25, 'reg_lambda': 1.2, 'scale_pos_weight': 4, 'subsample': 0.74}
=== Train ===
              precision    recall  f1-score   support

           0       0.93      0.43      0.58    150921
           1       0.36      0.91      0.51     52841

    accuracy                           0.55    203762
   macro avg       0.64      0.67      0.55    203762
weighted avg       0.78      0.55      0.56    203762

AUC PR: 0.45963513440725173
ROC AUC: 0.7429676429769856
=== validation ===
              precision    recall  f1-score   support

           0       0.93      0.43      0.58     37730
           1       0.36      0.90      0.51     13211

    accuracy                           0.55     50941
   macro avg       0.64      0.66      0.55     50941
weighted avg       0.78      0.55      0.56     50941

AUC PR: 0.4524557517140228
ROC AUC: 

In [117]:
model_full = grid_search.best_estimator_
model_full.fit(X, y)

y_pred_proba = model_full.predict_proba(test_df)[:, 1]

sample_submission = pd.read_csv('Data/sample_submission.csv')
sample_submission['probability'] = y_pred_proba
sample_submission.to_csv('./rsgg.csv', index=False)

d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:48:11] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
